In [5]:
#!/usr/bin/env python3
"""
Recreate IPTA MDC2 Group 2 Dataset 2 injection (Earth + pulsar terms)
using libstempo, with local constants and ecliptic→equatorial conversion.
"""

import numpy as np
import libstempo as lt
import os, glob

# ---- constants (units) ----
SOLAR2S = 4.925490947e-6          # G*M_sun/c^3 [s]
MPC2S   = 1.02927125e14           # 1 Mpc / c [s]
KPC2S   = 3.085677581e19 / 2.99792458e8  # 1 kpc / c [s]
EPS = np.deg2rad(23.439291111)    # obliquity of the ecliptic [rad]

def ecl_to_equ(elong, elat):
    """Tempo2 ELONG/ELAT (radians) -> RA,DEC (radians), IAU 2006 ε."""
    lam, beta = elong, elat
    sin_delta = np.sin(beta)*np.cos(EPS) + np.cos(beta)*np.sin(EPS)*np.sin(lam)
    delta = np.arcsin(sin_delta)
    y = np.sin(lam)*np.cos(EPS) - np.tan(beta)*np.sin(EPS)
    x = np.cos(lam)
    alpha = np.arctan2(y, x) % (2*np.pi)
    return alpha, delta  # RA, DEC

# ---- local add_cgw with RA/DEC OR ELONG/ELAT ----
def add_cgw(psr, gwtheta, gwphi, mc, dist, fgw, phase0, psi, inc,
            pdist=1.0, pphase=None, psrTerm=True, evolve=True,
            phase_approx=False, tref=0.0):
    mc   *= SOLAR2S
    dist *= MPC2S
    w0 = np.pi*fgw
    phase0 /= 2.0
    w053 = w0**(-5/3)

    cosgwtheta, cosgwphi = np.cos(gwtheta), np.cos(gwphi)
    singwtheta, singwphi = np.sin(gwtheta), np.sin(gwphi)
    sin2psi, cos2psi = np.sin(2*psi), np.cos(2*psi)
    incfac1, incfac2 = 0.5*(3 + np.cos(2*inc)), 2*np.cos(inc)

    m     = np.array([singwphi, -cosgwphi, 0.0])
    n     = np.array([-cosgwtheta*cosgwphi, -cosgwtheta*singwphi, singwtheta])
    omhat = np.array([-singwtheta*cosgwphi, -singwtheta*singwphi, -cosgwtheta])

    # --- pulsar sky position -> phat ---
    if ("RAJ" in psr.pars()) and ("DECJ" in psr.pars()):
        ra  = psr["RAJ"].val     # radians
        dec = psr["DECJ"].val    # radians
    elif ("ELONG" in psr.pars()) and ("ELAT" in psr.pars()):
        ra, dec = ecl_to_equ(psr["ELONG"].val, psr["ELAT"].val)
    else:
        raise ValueError(f"{psr.name} lacks RAJ/DECJ and ELONG/ELAT")

    ptheta = np.pi/2 - dec
    pphi   = ra
    phat = np.array([
        np.sin(ptheta)*np.cos(pphi),
        np.sin(ptheta)*np.sin(pphi),
        np.cos(ptheta)
    ])

    fplus  = 0.5 * ((m@phat)**2 - (n@phat)**2) / (1 + np.dot(omhat, phat))
    fcross =      ((m@phat) * (n@phat))        / (1 + np.dot(omhat, phat))
    cosMu  = -np.dot(omhat, phat)

    toas = psr.toas()*86400.0 - tref
    if pphase is not None:
        pd = pphase / (2*np.pi*fgw*(1 - cosMu)) / KPC2S
    else:
        pd = pdist
    pd *= KPC2S
    tp = toas - pd*(1 - cosMu)

    if evolve:
        fac1 = 256/5 * mc**(5/3) * w0**(8/3)
        fac2 = 1/32 / mc**(5/3)
        omega   = w0*(1 - fac1*toas)**(-3/8)
        omega_p = w0*(1 - fac1*tp  )**(-3/8)
        phase   = phase0 + fac2*(w053 - omega  **(-5/3))
        phase_p = phase0 + fac2*(w053 - omega_p**(-5/3))
    elif phase_approx:
        omega   = w0
        omega_p = w0*(1 + (256/5)*mc**(5/3)*w0**(8/3)*pd*(1 - cosMu))**(-3/8)
        phase   = phase0 + omega  * toas
        phase_p = phase0 + omega_p* toas
    else:
        omega = omega_p = w0
        phase   = phase0 + omega*toas
        phase_p = phase0 + omega*tp

    At   = np.sin(2*phase  )*incfac1
    Bt   = np.cos(2*phase  )*incfac2
    At_p = np.sin(2*phase_p)*incfac1
    Bt_p = np.cos(2*phase_p)*incfac2

    alpha   = mc**(5/3)/dist/omega  **(1/3)
    alpha_p = mc**(5/3)/dist/omega_p**(1/3)

    rplus   = alpha  *( At  *cos2psi + Bt  *sin2psi)
    rcross  = alpha  *(-At  *sin2psi + Bt  *cos2psi)
    rplus_p = alpha_p*( At_p*cos2psi + Bt_p*sin2psi)
    rcross_p= alpha_p*(-At_p*sin2psi + Bt_p*cos2psi)

    if psrTerm:
        res = fplus*(rplus_p - rplus) + fcross*(rcross_p - rcross)
    else:
        res = -fplus*rplus - fcross*rcross

    psr.stoas[:] += res/86400.0

# ---- paths ----
par_dir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par"
tim_dir = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/tim"
out_dir = "/scratch/na00078/projects/IPTA_MDC2/sims/G2D2_reinj"
os.makedirs(out_dir, exist_ok=True)

# ---- CW params (dataset 2) ----
gwtheta = 0.6387905062299246
gwphi   = 3.3335788713091694
mc      = 4.3e9
dist    = 75.4
fgw     = 3.7e-9
phase0  = 0.24434609527920614
psi     = 1.1187560505283651
inc     = 0.8412486994612669
tref = 55443.93364609394 * 86400  # seconds

# ---- inject and save ----
pars = sorted(glob.glob(os.path.join(par_dir, "*.par")))
for p in pars:
    name = os.path.basename(p)[:-4]
    t = os.path.join(tim_dir, f"{name}.tim")
    if not os.path.exists(t):
        continue

    psr = lt.tempopulsar(p, t)
    print(f"Injecting into {name} ...")
    add_cgw(psr, gwtheta, gwphi, mc, dist, fgw, phase0, psi, inc,
            psrTerm=True, evolve=True, tref=tref)

    out_tim = os.path.join(out_dir, f"{name}.tim")
    out_par = os.path.join(out_dir, f"{name}.par")
    psr.savetim(out_tim)
    os.system(f"cp {p} {out_par}")

print("Done. New CW-injected .tim files saved in:", out_dir)


/tmp/ipykernel_36312/1892960008.py:137: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  psr = lt.tempopulsar(p, t)


Injecting into J0030+0451 ...
[preProcess.C:158] Warning: PSR J0030+0451 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

Injecting into J0034-0534 ...
[preProcess.C:158] Warning: PSR J0034-0534 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

Injecting into J0218+4232 ...
[preProcess.C:158] Warning: PSR J0218+4232 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

Injecting into J0437-4715 ...
[preProcess.C:158] Warning: PSR J0437-4715 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org